# Fig5: station VRMSE around the closest approach

Compute station-wise vector RMSE for the three hours before and after each typhoon's closest approach to Hong Kong. The five 24 h AI-driven simulations are averaged in vector-component space before comparison with observations.

In [1]:
from __future__ import annotations

from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd

MODELS = ("pangu", "graphcast", "fengwu", "fuxi", "aurora")
CASES = (
    {
        "key": "ragasa",
        "label": "Ragasa",
        "center": pd.Timestamp("2025-09-23 22:00", tz="UTC"),
        "station_sheet": "Ragasa",
    },
    {
        "key": "yagi",
        "label": "Yagi",
        "center": pd.Timestamp("2024-09-05 12:00", tz="UTC"),
        "station_sheet": "Yagi",
    },
)
WINDOW_HALF_WIDTH = pd.Timedelta(hours=3)
EXPECTED_MINUTES = 6 * 60 + 1
TS_COL = "timestamp_utc"
TS_FORMAT = "%Y%m%d %H%M"
MAP_EXTENT = [113.824, 114.417, 22.139, 22.562]
FIGSIZE = (7.2, 6.2)
DPI = 600
CMAP_NAME = "viridis"

plt.rcParams.update({"font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 16})

In [2]:
def _find_fig5_dir() -> Path:
    """Locate Fig5 when running from Fig5 or the workspace root."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Fig5", cwd / "Figs" / "Fig5"]
    candidates.extend(parent / "Fig5" for parent in cwd.parents)
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "input/weather_station_location.xlsx").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate Fig5 from {cwd}")


def _read_station_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Missing station time series: {path}")
    df = pd.read_csv(path, dtype={TS_COL: str})
    if TS_COL not in df.columns:
        raise ValueError(f"Missing {TS_COL!r} in {path}")
    rename = {column: str(column).strip().lower() for column in df.columns if column != TS_COL}
    if len(rename.values()) != len(set(rename.values())):
        raise ValueError(f"Duplicate station columns after normalization in {path}")
    df = df.rename(columns=rename)
    df[TS_COL] = pd.to_datetime(
        df[TS_COL].astype(str).str.strip(), format=TS_FORMAT, utc=True, errors="raise"
    )
    if df[TS_COL].duplicated().any():
        raise ValueError(f"Duplicate timestamps in {path}")
    return df.sort_values(TS_COL).reset_index(drop=True)


def _load_uv_pair(
    speed_path: Path,
    direction_path: Path,
    start: pd.Timestamp,
    end: pd.Timestamp,
    expected_stations: list[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    speed = _read_station_csv(speed_path)
    direction = _read_station_csv(direction_path)
    speed_stations = [c for c in speed.columns if c != TS_COL]
    direction_stations = [c for c in direction.columns if c != TS_COL]
    if speed_stations != direction_stations:
        raise ValueError(f"Speed/direction station columns differ: {speed_path.parent}")
    stations = speed_stations if expected_stations is None else expected_stations
    if set(speed_stations) != set(stations):
        raise ValueError(f"Unexpected station set in {speed_path}")

    speed = speed.loc[speed[TS_COL].between(start, end), [TS_COL, *stations]].set_index(TS_COL)
    direction = direction.loc[direction[TS_COL].between(start, end), [TS_COL, *stations]].set_index(TS_COL)
    common = speed.index.intersection(direction.index)
    if len(common) != EXPECTED_MINUTES:
        raise RuntimeError(
            f"Expected {EXPECTED_MINUTES} common records in {speed_path.name}; found {len(common)}"
        )
    speed = speed.loc[common, stations].apply(pd.to_numeric, errors="coerce")
    direction = direction.loc[common, stations].apply(pd.to_numeric, errors="coerce")
    missing = speed.eq(32767) | direction.eq(32767) | (speed.eq(0.0) & direction.eq(0.0))
    speed = speed.mask(missing)
    direction = direction.mask(missing)
    theta = np.deg2rad(direction)
    u = -speed * np.sin(theta)
    v = -speed * np.cos(theta)
    return u, v


def _load_station_locations(path: Path, sheet_name: str, stations: list[str]) -> pd.DataFrame:
    loc = pd.read_excel(
        path, sheet_name=sheet_name, header=None, usecols=[0, 1, 2], names=["station", "lat", "lon"]
    )
    loc["station"] = loc["station"].astype(str).str.strip().str.lower()
    loc[["lat", "lon"]] = loc[["lat", "lon"]].apply(pd.to_numeric, errors="coerce")
    loc = loc.dropna(subset=["station", "lat", "lon"])
    if loc["station"].duplicated().any():
        raise ValueError(f"Duplicate station coordinates in sheet {sheet_name!r}")
    loc = loc.set_index("station")
    missing = sorted(set(stations) - set(loc.index))
    if missing:
        raise ValueError(f"Stations missing coordinates in {sheet_name!r}: {missing}")
    return loc.loc[stations].reset_index()


def _save_tiff(fig: mpl.figure.Figure, path: Path) -> None:
    try:
        fig.savefig(path, dpi=DPI, format="tiff", bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    except TypeError:
        fig.savefig(path, dpi=DPI, format="tiff", bbox_inches="tight")

In [3]:
fig5_dir = _find_fig5_dir()
input_dir = fig5_dir / "input"
output_dir = fig5_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
station_file = input_dir / "weather_station_location.xlsx"

case_results: dict[str, dict] = {}
for case in CASES:
    case_dir = input_dir / case["key"]
    start = case["center"] - WINDOW_HALF_WIDTH
    end = case["center"] + WINDOW_HALF_WIDTH
    obs_u, obs_v = _load_uv_pair(
        case_dir / "mean_wspd_obs.csv", case_dir / "mean_wdir_obs.csv", start, end
    )
    stations = list(obs_u.columns)

    model_u: list[pd.DataFrame] = []
    model_v: list[pd.DataFrame] = []
    for model in MODELS:
        sim_u, sim_v = _load_uv_pair(
            case_dir / f"{model}_mean_wspd_sim_24h.csv",
            case_dir / f"{model}_mean_wdir_sim_24h.csv",
            start,
            end,
            expected_stations=stations,
        )
        if not sim_u.index.equals(obs_u.index) or not sim_v.index.equals(obs_v.index):
            raise RuntimeError(f"Observation/model timestamps differ for {case['label']} {model}")
        model_u.append(sim_u)
        model_v.append(sim_v)

    ensemble_u = pd.DataFrame(
        np.nanmean(np.stack([frame.to_numpy() for frame in model_u]), axis=0),
        index=obs_u.index, columns=stations,
    )
    ensemble_v = pd.DataFrame(
        np.nanmean(np.stack([frame.to_numpy() for frame in model_v]), axis=0),
        index=obs_v.index, columns=stations,
    )
    squared_vector_error = (ensemble_u - obs_u) ** 2 + (ensemble_v - obs_v) ** 2
    vrmse = np.sqrt(squared_vector_error.mean(axis=0, skipna=True)).rename("vrmse")
    if not np.isfinite(vrmse.to_numpy()).all():
        raise RuntimeError(f"Non-finite station VRMSE for {case['label']}")

    locations = _load_station_locations(station_file, case["station_sheet"], stations)
    plot_df = locations.set_index("station").join(vrmse).reset_index()
    case_results[case["key"]] = {
        "case": case, "start": start, "end": end, "data": plot_df
    }
    print(
        f"{case['label']}: {start} to {end}; {len(obs_u)} minutes; "
        f"{len(stations)} stations; VRMSE {vrmse.min():.3f}-{vrmse.max():.3f} m/s"
    )

Ragasa: 2025-09-23 19:00:00+00:00 to 2025-09-24 01:00:00+00:00; 361 minutes; 30 stations; VRMSE 6.220-39.125 m/s


Yagi: 2024-09-05 09:00:00+00:00 to 2024-09-05 15:00:00+00:00; 361 minutes; 31 stations; VRMSE 3.259-14.600 m/s


In [4]:
cmap = mpl.colormaps[CMAP_NAME]

generated_paths: list[Path] = []
for key, result in case_results.items():
    case = result["case"]
    data = result["data"]
    values = data["vrmse"].to_numpy(dtype=float)
    case_min = float(np.nanmin(values))
    case_max = float(np.nanmax(values))
    if not np.isfinite(case_min) or not np.isfinite(case_max) or case_max <= case_min:
        raise RuntimeError(f"Invalid VRMSE range for {case['label']}: {case_min} to {case_max}")
    norm = mpl.colors.Normalize(vmin=case_min, vmax=case_max)
    sizes = 30.0 + 270.0 * (values / case_max) ** 1.5

    fig = plt.figure(figsize=FIGSIZE)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("10m"), facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN.with_scale("10m"), facecolor="white")
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.6, alpha=0.35, linestyle="--")
    gridlines.top_labels = False
    gridlines.right_labels = False
    scatter = ax.scatter(
        data["lon"], data["lat"], s=sizes, c=values, cmap=cmap, norm=norm,
        transform=ccrs.PlateCarree(), edgecolor="black", linewidth=0.5, alpha=0.90,
    )
    colorbar_ax = ax.inset_axes([1.02, 0.0, 0.035, 1.0], transform=ax.transAxes)
    colorbar = fig.colorbar(scatter, cax=colorbar_ax)
    colorbar.set_label("VRMSE (m s$^{-1}$)")
    colorbar.locator = MaxNLocator(integer=True)
    colorbar.update_ticks()

    start_tag = result["start"].strftime("%Y%m%d-%H")
    end_tag = result["end"].strftime("%Y%m%d-%H")
    out_path = output_dir / f"vrmse_station_spatial_{key}_{start_tag}_to_{end_tag}.tif"
    _save_tiff(fig, out_path)
    plt.close(fig)
    generated_paths.append(out_path)
    print(f"Saved: {out_path} (scale {case_min:.3f}-{case_max:.3f} m/s)")

if len(generated_paths) != 2 or any(not path.is_file() or path.stat().st_size == 0 for path in generated_paths):
    raise RuntimeError("Expected two non-empty VRMSE spatial TIFF files")
print("Done.")

Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\vrmse_station_spatial_ragasa_20250923-19_to_20250924-01.tif (scale 6.220-39.125 m/s)


Saved: E:\BaiduSyncdisk\Code\06_AI_WRF_UCM\Figs\Fig5\output\vrmse_station_spatial_yagi_20240905-09_to_20240905-15.tif (scale 3.259-14.600 m/s)
Done.
